# SWE015 · Use Case 4 – Real-Time Chat Moderation (Turkish)
Trains and evaluates all models on Colab.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Then run the cells top to bottom.
Every number reported in the presentation comes from `results/results.json` produced here.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Get the code
GitHub user and Hugging Face repo of this project.

In [ ]:
GITHUB_USER = "bgmzyd"
HF_REPO_ID = "bgmzydd/chat-moderation-tr-models"   # where the weights will be uploaded

import os
!git clone https://github.com/{GITHUB_USER}/chat-moderation-tr.git
%cd chat-moderation-tr
!pip install -q -r requirements.txt
os.environ["HF_REPO_ID"] = HF_REPO_ID

## 2 · Data: download OffensEval-TR 2020 and create the fixed splits

In [ ]:
!python -m src.prepare_data

## 3 · Area B – edit-distance lexicon rule baseline and tf-idf baseline

In [ ]:
!python -m src.lexicon

In [ ]:
!python -m src.train_tfidf

## 4 · Area A – own word2vec + feed-forward network (SGD + backprop)

In [ ]:
!python -m src.train_w2v_ffn

## 5 · Area C – fine-tune BERTurk (GPU, ~10 min)

In [ ]:
!python -m src.train_bert

## 6 · Area D – quantization: PyTorch dynamic INT8 and ONNX Runtime INT8

In [ ]:
!python -m src.quantize_bert

In [ ]:
!python -m src.quantize_onnx

## 7 · Evaluate all 6 models on the same test set (CPU latency, ≥100 runs)

In [ ]:
!python -m src.evaluate

In [ ]:
from IPython.display import Markdown
Markdown(open("results/results_table.md").read())

## 8 · Upload the weights to the Hugging Face Hub (public model repo)
Run the login cell, open https://hf.co/oauth/device, enter the code it shows and authorize **write** access.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python -m src.hub upload --repo $HF_REPO_ID

## 9 · Download the results folder (commit it to GitHub)

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

## 10 · (optional) try the app here with a public link

In [ ]:
!python app.py --share